# 08 — 60-Second Per-Host Windowed Behavioral Features

**Goal:** Instead of classifying one isolated `conn.log` row at a time, group each host's traffic into **60-second windows** (`src_ip` + 1-minute bucket) and summarize its behavior.

**Why no `proto` or `dest_port` features here?**
In UWF, benign traffic is mostly UDP/DNS and attacks are mostly TCP. Leaving `proto` and `dest_port` out forces the model to learn **real host behavior** (connection bursts, port/IP fan-out, failure rates, byte ratios) rather than a lab shortcut.

**Note on `TON_IoT`:** `train_test_network.csv` stripped the `ts` timestamp column, so we cannot build 60-second windows on it. Here we train on timestamped 2024 Zeek captures and test on unseen future 2025 captures:
- **Run 1:** `UWF-ZeekData24` only
- **Run 2:** `UWF-ZeekData24` + `UWF-ZeekDataFall24-2`
- **Tested on unseen 2025:** `UWF-ZeekDataSum25-1` and `UWF-ZeekDataSum25-2`

In [1]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SEED = 42
WINDOW_SEC = 60
FAILED_STATES = {"S0", "REJ", "RSTO", "RSTR", "SH", "SHR", "RSTOS0", "RSTRH"}

WINDOW_FEATURES = [
    "conn_count",
    "unique_dst_ips",
    "unique_dst_ports",
    "ports_per_ip",
    "failed_ratio",
    "sf_ratio",
    "mean_duration",
    "total_orig_bytes",
    "total_resp_bytes",
    "bytes_ratio",
    "mean_orig_pkts",
    "mean_resp_pkts",
]


def load_uwf_windows(dataset_name: str, window_sec: int = WINDOW_SEC) -> pd.DataFrame:
    """Load a UWF Zeek Parquet dataset and aggregate flows into per-host 60s behavioral windows."""
    pattern = str(ROOT / f"datasets/uwf_zeek/{dataset_name}/parquet/*/*.parquet")
    files = sorted(glob.glob(pattern))
    cols = [
        "ts", "src_ip_zeek", "dest_ip_zeek", "dest_port_zeek", "conn_state",
        "duration", "orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts",
        "label_binary", "label_tactic",
    ]
    df = pd.concat([pd.read_parquet(f, columns=cols) for f in files], ignore_index=True)
    df = df[df["label_binary"] != "Duplicate"].copy()

    for c in ["duration", "orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts"]:
        df[c] = df[c].fillna(0.0).astype(float)

    df["is_attack"] = (df["label_binary"] == "True").astype(int)
    df["is_failed"] = df["conn_state"].isin(FAILED_STATES).astype(float)
    df["is_sf"] = (df["conn_state"] == "SF").astype(float)
    df["window_id"] = (df["ts"] // window_sec).astype("int64")

    # Group by (src_ip, 60-second bucket)
    grp = df.groupby(["src_ip_zeek", "window_id"], sort=False)
    win = grp.agg(
        conn_count=("ts", "size"),
        unique_dst_ips=("dest_ip_zeek", "nunique"),
        unique_dst_ports=("dest_port_zeek", "nunique"),
        failed_ratio=("is_failed", "mean"),
        sf_ratio=("is_sf", "mean"),
        mean_duration=("duration", "mean"),
        total_orig_bytes=("orig_bytes", "sum"),
        total_resp_bytes=("resp_bytes", "sum"),
        mean_orig_pkts=("orig_pkts", "mean"),
        mean_resp_pkts=("resp_pkts", "mean"),
        label=("is_attack", "max"),  # 1 if ANY attack flow happened in this host-minute
        attack_flows=("is_attack", "sum"),
    ).reset_index()

    win["ports_per_ip"] = win["unique_dst_ports"] / win["unique_dst_ips"].clip(lower=1)
    win["bytes_ratio"] = win["total_orig_bytes"] / (win["total_orig_bytes"] + win["total_resp_bytes"] + 1.0)

    # Dominant attack tactic in each attack window (for per-tactic evaluation)
    atk_only = df[df["is_attack"] == 1]
    dom_tactic = (
        atk_only.groupby(["src_ip_zeek", "window_id"])["label_tactic"]
        .agg(lambda s: s.value_counts().index[0])
        .rename("dominant_tactic")
    )
    win = win.merge(dom_tactic, on=["src_ip_zeek", "window_id"], how="left")
    win["dominant_tactic"] = win["dominant_tactic"].fillna("none")
    return win


In [2]:
datasets = {}
for name in ["UWF-ZeekData24", "UWF-ZeekDataFall24-2", "UWF-ZeekDataSum25-1", "UWF-ZeekDataSum25-2"]:
    datasets[name] = load_uwf_windows(name)

pd.DataFrame([
    {
        "Dataset": name,
        "Host-Minute Windows": len(w),
        "Benign Windows": int((w["label"] == 0).sum()),
        "Attack Windows": int((w["label"] == 1).sum()),
    }
    for name, w in datasets.items()
])

,Dataset,Host-Minute Windows,Benign Windows,Attack Windows
0,UWF-ZeekData24,174686,152984,21702
1,UWF-ZeekDataFall24-2,517350,517209,141
2,UWF-ZeekDataSum25-1,293743,291853,1890
3,UWF-ZeekDataSum25-2,161617,161585,32


In [3]:
# Compare median host-window behavior: Benign vs Attack (in UWF-ZeekData24)
w24 = datasets["UWF-ZeekData24"]
print("=== Median 60-second host behavior in UWF-ZeekData24 ===")
display(w24.groupby("label")[WINDOW_FEATURES].median().T.rename(columns={0: "Benign", 1: "Attack"}).round(3))

print("\n=== Sample Attack Windows (notice conn_count, unique_dst_ports, failed_ratio) ===")
display(w24[w24["label"] == 1][["dominant_tactic"] + WINDOW_FEATURES[:7]].head(8))

=== Median 60-second host behavior in UWF-ZeekData24 ===


label,Benign,Attack
conn_count,2.000,5.000
unique_dst_ips,1.000,1.000
unique_dst_ports,1.000,1.000
ports_per_ip,1.000,1.000
failed_ratio,1.000,0.875
sf_ratio,0.000,0.000
mean_duration,0.004,0.093
total_orig_bytes,188.000,568.000
total_resp_bytes,0.000,201.000
bytes_ratio,0.985,0.053



=== Sample Attack Windows (notice conn_count, unique_dst_ports, failed_ratio) ===


,dominant_tactic,conn_count,unique_dst_ips,unique_dst_ports,ports_per_ip,failed_ratio,sf_ratio,mean_duration
0,Credential Access,336,1,2,2.0,0.005952,0.994048,1.284693
1,Defense Evasion,1,1,1,1.0,1.000000,0.000000,0.000954
2,Defense Evasion,1,1,1,1.0,1.000000,0.000000,0.001079
3,Defense Evasion,1,1,1,1.0,1.000000,0.000000,0.001093
4,Defense Evasion,1,1,1,1.0,1.000000,0.000000,0.001094
5,Defense Evasion,1,1,1,1.0,1.000000,0.000000,0.001114
6,Defense Evasion,1,1,1,1.0,1.000000,0.000000,0.001161
7,Defense Evasion,1,1,1,1.0,1.000000,0.000000,0.001173


In [4]:
runs = {
    "Run 1: UWF-24 only": datasets["UWF-ZeekData24"],
    "Run 2: UWF-24 + Fall24-2": pd.concat(
        [datasets["UWF-ZeekData24"], datasets["UWF-ZeekDataFall24-2"]], ignore_index=True
    ),
}

test_sets = {
    "UWF-Sum25-1 (unseen 2025)": datasets["UWF-ZeekDataSum25-1"],
    "UWF-Sum25-2 (unseen 2025)": datasets["UWF-ZeekDataSum25-2"],
}

fitted = {}
rows = []
for run_name, train_w in runs.items():
    clf = HistGradientBoostingClassifier(random_state=SEED, class_weight="balanced")
    clf.fit(train_w[WINDOW_FEATURES], train_w["label"].to_numpy(dtype=int))
    fitted[run_name] = clf

    for test_name, test_w in test_sets.items():
        y_true = test_w["label"].to_numpy(dtype=int)
        y_pred = clf.predict(test_w[WINDOW_FEATURES])
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        rows.append({
            "Run": run_name,
            "Test Set": test_name,
            "Windows": len(test_w),
            "F1": round(f1_score(y_true, y_pred), 4),
            "Precision": round(precision_score(y_true, y_pred), 4),
            "Recall": round(recall_score(y_true, y_pred), 4),
            "FPR": round(fp / (fp + tn), 4),
        })

results_df = pd.DataFrame(rows)
results_df

,Run,Test Set,Windows,F1,Precision,Recall,FPR
0,Run 1: UWF-24 only,UWF-Sum25-1 (unseen 2025),293743,0.6857,0.8846,0.5598,0.0005
1,Run 1: UWF-24 only,UWF-Sum25-2 (unseen 2025),161617,0.0004,0.0002,0.0312,0.0281
2,Run 2: UWF-24 + Fall24-2,UWF-Sum25-1 (unseen 2025),293743,0.9716,0.9645,0.9788,0.0002
3,Run 2: UWF-24 + Fall24-2,UWF-Sum25-2 (unseen 2025),161617,0.0279,0.0141,0.9375,0.0129


In [5]:
# 1. What features is Run 2 actually using?
clf2 = fitted["Run 2: UWF-24 + Fall24-2"]
test_s1 = test_sets["UWF-Sum25-1 (unseen 2025)"]
imp = permutation_importance(
    clf2, test_s1[WINDOW_FEATURES], test_s1["label"].to_numpy(dtype=int),
    n_repeats=5, random_state=SEED, scoring="f1", n_jobs=-1
)
imp_df = pd.DataFrame({
    "Feature": WINDOW_FEATURES,
    "F1 drop when shuffled": imp.importances_mean.round(4),
}).sort_values("F1 drop when shuffled", ascending=False)
print("=== Permutation Importance on Unseen 2025 (Sum25-1) ===")
display(imp_df)

# 2. Per-tactic recall on unseen 2025 windows
for test_name, test_w in test_sets.items():
    preds = clf2.predict(test_w[WINDOW_FEATURES])
    tacs = test_w["dominant_tactic"].to_numpy(dtype=str)
    trows = []
    for t in sorted(np.unique(tacs)):
        m = tacs == t
        trows.append({
            "Tactic": "none (benign FPR)" if t == "none" else t,
            "Windows": int(m.sum()),
            "Flagged Rate": round(float((preds[m] == 1).mean()), 4),
        })
    print(f"\n=== Run 2 per-tactic breakdown on {test_name} ===")
    display(pd.DataFrame(trows))

=== Permutation Importance on Unseen 2025 (Sum25-1) ===


,Feature,F1 drop when shuffled
11,mean_resp_pkts,0.9140
10,mean_orig_pkts,0.7428
6,mean_duration,0.0824
4,failed_ratio,0.0425
0,conn_count,0.0385
7,total_orig_bytes,0.0032
9,bytes_ratio,0.0017
1,unique_dst_ips,0.0000
3,ports_per_ip,0.0000
2,unique_dst_ports,0.0000



=== Run 2 per-tactic breakdown on UWF-Sum25-1 (unseen 2025) ===


,Tactic,Windows,Flagged Rate
0,Defense Evasion,14,1.0000
1,Discovery,3,1.0000
2,Initial Access,1,1.0000
3,Lateral Movement,15,0.8667
4,Reconnaissance,1857,0.9795
5,none (benign FPR),291853,0.0002



=== Run 2 per-tactic breakdown on UWF-Sum25-2 (unseen 2025) ===


,Tactic,Windows,Flagged Rate
0,Discovery,20,1.0000
1,Reconnaissance,12,0.8333
2,none (benign FPR),161585,0.0129


### Key Takeaways
- **Real-world rarity & Base-Rate Fallacy:** Grouping 1.87M flows into host-minutes collapses `UWF-Sum25-2` to only **32 attack windows** out of 161,617 (0.02%). Even a 1.29% FPR produces ~2,084 false alarms vs. 30 true attacks.
- **Hidden lab shortcut:** Because `UWF` benign windows have median `0` response packets, the model relied on `mean_resp_pkts` instead of `unique_dst_ports`.
- **Next (`09`):** Convert real enterprise LAN PCAPs (`CIC-IDS2017`, 5 days) with Zeek and test a **Hybrid** model (per-flow + 60s host context) so we keep full row resolution without blurring stealthy attacks.